# TrainLM: native TorchTPU trainer smoke

Open in Colab on a single-device Cloud TPU v5e runtime. One code cell clones a fresh source snapshot; the next runs a three-step `distilgpt2` training and export smoke in a child process. The optional resume mode compares an additional four-step reference run. This is experimental evidence, not performance or support certification.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-tpu-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "feat/01-native-torchtpu-single-device-training",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## Run the trainer

The default runs eager training for three steps. Set `RUN_COMPILE = True` only after eager succeeds. Set `RUN_RESUME = True` to save, resume for one step, and require exact model-weight agreement with an uninterrupted four-step run. Resume requires eager mode. The child has a timeout and process-group cleanup; raw logs stay in the local session directory. A bounded JSON evidence report is saved separately for sharing.

In [ ]:
import json
import runpy

RUN_COMPILE = False
RUN_RESUME = False
if RUN_COMPILE and RUN_RESUME:
    raise ValueError("Run compiled and resume checks separately.")
TIMEOUT_SECONDS = 1800 if (RUN_COMPILE or RUN_RESUME) else 900
run_process = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
session_dir = Path(tempfile.mkdtemp(prefix="trainlm-native-run-"))
command = [sys.executable, str(repo_root / "examples/torchtpu_native_trainer_smoke.py"),
           "--output-dir", str(session_dir / "artifacts")]
if RUN_COMPILE:
    command.append("--compile")
if RUN_RESUME:
    command.append("--resume")
result = run_process(command, cwd=repo_root, timeout=TIMEOUT_SECONDS)
(session_dir / "stdout.log").write_text(result["stdout"])
(session_dir / "stderr.log").write_text(result["stderr"])
build_report = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_trainer_evidence.py"))["build_trainer_report"]
report = build_report(result, source_revision=source_revision,
                      compile_requested=RUN_COMPILE, resume_requested=RUN_RESUME)
report_path = session_dir / "trainlm_native_trainer_report.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
print(json.dumps(report, indent=2))
print(f"shareable_report={report_path}")
print(f"local_artifacts_and_logs={session_dir}")
if report["status"] != "passed":
    raise RuntimeError("Native trainer smoke failed; inspect local logs before retrying.")

## Download evidence

Run this cell after the trainer cell, including when that cell reports a failed check. Share only the JSON report; raw logs may contain local details.

In [ ]:
from google.colab import files
files.download(str(report_path))

## If it fails

Keep the first error in the local logs and the source revision. A detected TPU must not be labeled a CPU pass. Torch 2.14.x remains outside TrainLM's declared package range; the owner-run CPU suite and native package profile must be reviewed before changing it.